# S50_02 — Inference Servers

Running LLMs in production requires an inference server that handles batching, concurrency, memory management, and OpenAI-compatible APIs.

## Key inference servers

| Server | Best for | Key feature |
|--------|---------|-------------|
| **vLLM** | Production GPU serving | PagedAttention — highest throughput |
| **Ollama** | Local development | One-command setup, GGUF models |
| **llama.cpp** | CPU / low VRAM | GGUF, runs on MacBook |
| **TGI** (HuggingFace) | Production | OpenAI-compatible, Docker |
| **LM Studio** | Desktop GUI | Non-technical users |
| **Triton Inference Server** | NVIDIA production | Multi-model, GPU optimization |

## Ollama — local development

In [ ]:
# Ollama setup:
# 1. Download from ollama.com
# 2. ollama pull llama3.2:1b
# 3. ollama serve (or it runs as a background service)

# Ollama exposes an OpenAI-compatible API at http://localhost:11434
from openai import OpenAI

# Point to Ollama instead of OpenAI
ollama_client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',  # any string
)

# Identical call to OpenAI's API
# response = ollama_client.chat.completions.create(
#     model='llama3.2:1b',
#     messages=[{'role': 'user', 'content': 'What is gradient descent?'}],
#     max_tokens=256,
# )
# print(response.choices[0].message.content)

print('Ollama commands:')
print('  ollama pull llama3.2:1b     # download model')
print('  ollama pull mistral         # another model')
print('  ollama list                 # show downloaded models')
print('  ollama run llama3.2:1b      # interactive chat')
print('  ollama serve                # start API server')

## vLLM — production throughput

In [ ]:
# vLLM uses PagedAttention — manages KV cache like OS pages
# Result: 10-20x higher throughput than naive implementation

# Start vLLM server:
# pip install vllm
# python -m vllm.entrypoints.openai.api_server \
#     --model meta-llama/Llama-3.2-1B-Instruct \
#     --port 8000 \
#     --dtype bfloat16

# Also OpenAI-compatible
# vllm_client = OpenAI(base_url='http://localhost:8000/v1', api_key='vllm')

# vLLM Python API (in-process)
vllm_offline = '''
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Llama-3.2-1B-Instruct", dtype="bfloat16")

prompts = [
    "What is the capital of France?",
    "Explain overfitting in one sentence.",
    "Write a Python function to reverse a string.",
]

params = SamplingParams(temperature=0.7, max_tokens=256)
outputs = llm.generate(prompts, params)  # batches all prompts automatically

for output in outputs:
    print(f"Prompt: {output.prompt[:50]}")
    print(f"Output: {output.outputs[0].text[:100]}")
    print()
'''
print('vLLM offline batch inference:')
print(vllm_offline)

## Continuous batching and throughput

In [ ]:
import time
import anthropic

client = anthropic.Anthropic()

# Demonstrate streaming vs non-streaming latency perception
prompt = 'List 5 key concepts in machine learning, one per line.'

# Non-streaming: wait for full response
start = time.time()
msg = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=256,
    messages=[{'role': 'user', 'content': prompt}],
)
full_latency = time.time() - start
print(f'Non-streaming total latency: {full_latency:.2f}s')
print(msg.content[0].text[:200])

print()

# Streaming: first token arrives much faster
start = time.time()
first_token_time = None
with client.messages.stream(
    model='claude-haiku-4-5-20251001',
    max_tokens=256,
    messages=[{'role': 'user', 'content': prompt}],
) as stream:
    for text in stream.text_stream:
        if first_token_time is None:
            first_token_time = time.time() - start
        break  # just measure time-to-first-token

print(f'Streaming TTFT (time to first token): {first_token_time:.2f}s')
print('→ Streaming gives lower perceived latency even if total time is the same')

## Choosing an inference setup

```
Development / experimentation?
  → Ollama (easiest setup, runs on laptop)

Serving to a small team (<10 concurrent users)?
  → vLLM on a single GPU, or TGI on Docker

High-traffic production?
  → vLLM with multiple replicas behind a load balancer
  → Or: use a managed API (Anthropic/OpenAI/Groq) and skip the infra

Edge / mobile?
  → llama.cpp (C++ binaries), MLC-LLM, or Apple MLX on Mac
```

Next: [S50_03_llm_monitoring.ipynb](./S50_03_llm_monitoring.ipynb)